In [1]:
import pandas as pd
import duckdb
import os
import glob

In [2]:
#read files into dictionary
dfs = {}
for file in glob.glob(os.path.join("clean_data", "*.parquet")): #search for all parquet files in clean_data, and iterate
    name = os.path.basename(file).replace(".parquet", "")
    dfs[name] = pd.read_parquet(file)

ANALYSIS 1: HAVE SPECIALISED MOLDS REPLACED ASSEMBLIES OF SMALLER PARTS TO CREATE THE SAME SHAPE? 
...OR VICE VERSA?

THE 'PART_RELATIONSHIPS' TABLE SHOWS 'PARENT' PARTS THAT HAVE RELATED 'CHILD' PARTS.
THESE 'CHILD' PARTS ARE EITHER SEGMENTS THAT COMBINE TO MAKE UP THE PARENT, 
OR THEY ARE A COMINBATION OF MULTIPLE child PARTS.
EITHER WAY, THE child(S) ARE THE PART(S) THAT WERE RELEASED FIRST.

TABLES NEEDED (6 out of 12):

    part_relationships - core table to identify child-child connections
    parts - to get the part name, for human readability
    part_categories - allow for more granular analysis by part category
    inventory_parts, inventories, sets - the 'year' column in the 'sets' table is needed to get the timeline of parent and child part usage


In [3]:
part_relationships = dfs["part_relationships"]
parts = dfs["parts"]
part_categories = dfs["part_categories"]
inventory_parts = dfs["inventory_parts"]
inventories = dfs["inventories"]
inventory_sets = dfs["inventory_sets"]
sets = dfs["sets"]

In [4]:
#only look at the parts that have a parent-child relatiosnhip, not eg. a mold update or print
part_relationships = duckdb.sql("SELECT * FROM part_relationships WHERE rel_type = 'R'").df()

In [5]:
#joined table is symmetrical in structure, where parent part info starts from the middle and continues through the left columns,
#and child part indo starts from the middle and continues through the right columnd
parent_child_parts = duckdb.sql("""
                    
                    WITH parts_sets AS(
                        SELECT
                            p.part_num,
                            s.set_num,
                            s.year,
                            ivs.quantity
                        FROM parts p JOIN inventory_parts ip ON p.part_num = ip.part_num
                        JOIN inventories i ON ip.inventory_id = i.id
                        JOIN inventory_sets ivs ON i.set_num = ivs.set_num
                        JOIN sets s ON ivs.set_num = s.set_num
                    )

                    SELECT DISTINCT ON(pr.parent_part_num, pr.child_part_num)
                                
                        pr.parent_part_num || pr.child_part_num AS id,
                        
                        MIN(parent_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS min_parent_year,
                        MAX(parent_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS max_parent_year,
                        pr.parent_part_num,
                    
                        pr.child_part_num,
                        MIN(child_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS min_child_year,
                        MAX(child_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS max_child_year

                    FROM part_relationships pr

                    JOIN parts child_p ON pr.child_part_num = child_p.part_num
                    JOIN parts parent_p ON pr.parent_part_num = parent_p.part_num

                    JOIN parts_sets parent_ps ON parent_p.part_num = parent_ps.part_num
                    JOIN parts_sets child_ps ON child_p.part_num = child_ps.part_num

                    WHERE pr.rel_type = 'R' --only ~8 percent of all parts have this relationship
                                
                    ORDER BY pr.parent_part_num, parent_ps.year ASC, child_ps.year ASC
                    
                    """).df() #takes 3.2 - 3.3 seconds to executed

In [6]:
parent_child_parts


,id,min_parent_year,max_parent_year,parent_part_num,child_part_num,min_child_year,max_child_year
0,1112611767,2014,2022,11126,11767,2014,2014
1,1112671100,2014,2022,11126,71100,2022,2022
2,1120811209,2014,2024,11208,11209,2014,2023
3,112085033,2014,2024,11208,5033,2024,2024
4,1121361485,2013,2025,11213,61485,2008,2026
...,...,...,...,...,...,...,...
973,upn0027pr0001upn0026pr0001,1992,1992,upn0027pr0001,upn0026pr0001,1992,1992
974,upn0027pr000171308,1992,1992,upn0027pr0001,71308,1992,1992
975,upn003571094,1990,1995,upn0035,71094,1990,1995
976,upn0068upn0182,2013,2013,upn0068,upn0182,2013,2013


- DISCOVERY: ACROSS THE BOARD, THE DATABSSE DOES NOT CONSIDER TWO PARTS THAT STRUCTURALLY COMBINE TO MAKE THE THIRD AS A 'PARENT-CHILD' RELATIONSHIP 
    - IN FACT, THERE IS NO RELATIOSNHIP WHATSOEVER. EG. BETWEEN PART 51739, AND 24299 WITH 24307. THIS IS LIKELY BECAUSE THE THE CONNECTIONS UNDERNEATH THE PIECES DIFFER. 
- FIX: I WILL HAVE TO USE MY OWN KNOWLEDGE OF PARTS TO PIECE TOGETHER MY ANALYSIS

In [7]:
mydf = duckdb.sql("SELECT * FROM part_relationships WHERE parent_part_num = '51739' OR child_part_num = '51739'")
mydf

┌──────────┬────────────────┬─────────────────┐
│ rel_type │ child_part_num │ parent_part_num │
│ varchar  │    varchar     │     varchar     │
└──────────┴────────────────┴─────────────────┘
                    0 rows                   

- DISCOVERY: SOME SPECIFIC INSTANCES OF PARENT-CHILD PART RELATIONSHIPS ARE PRESENT AFTER SOME MANUAL INSPECTIONS
- FIX: EXTRACT THE MAJORITY BY EXCLUDING WHERE MINIMUM PARENT YEAR = MINIMUM CHILD YEAR...
    - ...BECAUSE PARTS THAT WERE CREATED IN THE SAME YEAR ARE LIKELY TO BE PART MIRRORINGS, RATHER THAN 'TRUE' PARENT AND CHILD PARTS

In [8]:
parent_child_parts = duckdb.sql("SELECT * FROM parent_child_parts WHERE min_parent_year != min_child_year").df()
parent_child_parts

,id,min_parent_year,max_parent_year,parent_part_num,child_part_num,min_child_year,max_child_year
0,1112671100,2014,2022,11126,71100,2022,2022
1,112085033,2014,2024,11208,5033,2024,2024
2,1121361485,2013,2025,11213,61485,2008,2026
3,1121327448,2013,2025,11213,27448,2019,2025
4,1129987913,2013,2016,11299,87913,2010,2016
...,...,...,...,...,...,...,...
480,98374pr000195344,2013,2013,98374pr0001,95344,2011,2024
481,98374pr000295344,2015,2015,98374pr0002,95344,2011,2024
482,98374pr000395344,2016,2016,98374pr0003,95344,2011,2024
483,9845961649,2010,2018,98459,61649,2007,2018


In [9]:
#for each row of parent and child parts, find the quantity of each part used across all sets, for each year that part was in production

part_qty_per_year = duckdb.sql(
""" 
    WITH generic_part_totals AS( --this CTE is similar to my orignal part_qty_per_year
        SELECT 
            ip.part_num, -- 'parts' table not needed as part_num is in 'inventory_parts'!
            s.year,
            SUM(ip.quantity)::int AS quantity
        FROM inventory_parts ip
        JOIN inventories i ON ip.inventory_id = i.id
        JOIN sets s ON i.set_num = s.set_num
        GROUP BY ip.part_num, s.year
    ),

    valid_years_per_relationship AS(
        SELECT DISTINCT ON (pcp.parent_part_num, pcp.child_part_num, gpt.year)
            pcp.parent_part_num,
            pcp.child_part_num,
            gpt.year
        FROM parent_child_parts pcp JOIN generic_part_totals gpt
        ON pcp.parent_part_num = gpt.part_num OR pcp.child_part_num = gpt.part_num   --the DISTINCT will discard the 
    )

    --combining 'generic_part_totals' and 'valid_years_per_relationship':

    SELECT

        vy.parent_part_num || vy.child_part_num AS id, 

        vy.year,

        vy.parent_part_num,
        vy.child_part_num,

        COALESCE(parent_pt.quantity, 0) AS parent_qty,
        COALESCE(child_pt.quantity, 0) AS child_qty -- coalesce handles gaps in overlapping part timelines 

    FROM valid_years_per_relationship vy

    LEFT JOIN generic_part_totals parent_pt ON vy.parent_part_num = parent_pt.part_num
    AND vy.year = parent_pt.year

    LEFT JOIN generic_part_totals child_pt ON vy.child_part_num = child_pt.part_num
    AND vy.year = child_pt.year

    ORDER BY parent_part_num, child_part_num, year

"""
).df()

part_qty_per_year


,id,year,parent_part_num,child_part_num,parent_qty,child_qty
0,1112671100,2013,11126,71100,25,0
1,1112671100,2014,11126,71100,13,0
2,1112671100,2022,11126,71100,4,4
3,112085033,2013,11208,5033,12,0
4,112085033,2014,11208,5033,16,0
...,...,...,...,...,...,...
11397,flex08c126644,1999,flex08c12,6644,0,6
11398,flex08c126644,2000,flex08c12,6644,0,6
11399,flex08c126644,2001,flex08c12,6644,0,2
11400,flex08c126644,2003,flex08c12,6644,2,4


In [17]:
part_names_categories= duckdb.sql("""
                             
SELECT p.part_num, p.name, pc.name AS category
FROM parts p JOIN part_categories pc ON p.part_cat_id = pc.id
WHERE p.part_num IN(SELECT parent_part_num FROM part_qty_per_year)
OR p.part_num IN(SELECT child_part_num FROM part_qty_per_year)

""").df()
                             
part_names_categories
                             

,part_num,name,category
0,100559pat0001pr0002,"Animal, Dog, Dachshund with Vibrant Yellow Har...",Animals / Creatures
1,11126,Rip Cord Flexible with Handle,Tools
2,11208,"Wheel 14mm D. x 9.9mm with Centre Groove, Fake...",Wheels and Tyres
3,11213,Plate Round 6 x 6 with Hole,Plates Round Curved and Dishes
4,11299,Ladder 16 x 3.5 with Side Supports,"Bars, Ladders and Fences"
...,...,...,...
460,98459,"Duplo Door / Lid, Wood Effect","Duplo, Quatro and Primo"
461,98562,Large Figure Weapon Claw / Handcuff,Large Buildable Figures
462,98563,"Large Figure Weapon, Zamor Sphere Launcher, To...",Large Buildable Figures
463,flex08c12,Technic Flex Cable 12L,Technic Special


In [19]:
os.makedirs("report_data", exist_ok=True)
parent_child_parts.to_csv(os.path.join("report_data", "parent_child_parts.csv"), index=False)
part_qty_per_year.to_csv(os.path.join("report_data", "part_qty_per_year"), index=False)
part_names_categories.to_csv(os.path.join("report_data", "part_names_categories.csv"), index=False)